In [148]:
import os 
import numpy as np 
import pandas as pd 
from absl import app 
from absl import flags 
import compress_pickle
from typing import Sequence 
import matplotlib.pyplot as plt

In [149]:
params_list = [] 
all_params_list = [] 
ks = [x for x in range(1,11)] 
# ks = [x for x in range(120,501, 20)]
# ks.extend([x for x in range(20,1001, 20)])
ks.extend([x for x in range(20,501, 20)])
# ks.extend([x for x in range(20,101, 20)])

def get_n_k_for_num_ratings(all_values, num_ratings=5000): 
    values=[] 
    for x in all_values:
        if x==0:
            x=1 
        n = int(np.floor(num_ratings/x)) 
        if n > 0:
            values.append((n, int(x))) 
    return values


# vals = get_n_k_for_num_ratings(ks) 
# [x[1] for x in vals], [x[0] for x in vals]
ratings_list = []
# nk_list = [2500]
# nk_list = [5000]
nk_list = [2500, 5000, 10000, 25000, 50000]
# nk_list = [1000, 2500, 5000, 10000, 25000, 50000]
# nk_list = [100, 250, 500, 1000, 2500]
# nk_list = [100, 250, 500, 1000, 2500, 5000, 10000, 25000, 50000]
# for r in range(1000, 5001, 1000):
# for r in range(5000, 5001, 500): 
for r in nk_list:
    params = get_n_k_for_num_ratings(ks, r)
    params_list.append(params)
    all_params_list.extend(params)
    ratings_list.extend([(r-x[0]*x[1]) for x in params])
    print(params)

[(2500, 1), (1250, 2), (833, 3), (625, 4), (500, 5), (416, 6), (357, 7), (312, 8), (277, 9), (250, 10), (125, 20), (62, 40), (41, 60), (31, 80), (25, 100), (20, 120), (17, 140), (15, 160), (13, 180), (12, 200), (11, 220), (10, 240), (9, 260), (8, 280), (8, 300), (7, 320), (7, 340), (6, 360), (6, 380), (6, 400), (5, 420), (5, 440), (5, 460), (5, 480), (5, 500)]
[(5000, 1), (2500, 2), (1666, 3), (1250, 4), (1000, 5), (833, 6), (714, 7), (625, 8), (555, 9), (500, 10), (250, 20), (125, 40), (83, 60), (62, 80), (50, 100), (41, 120), (35, 140), (31, 160), (27, 180), (25, 200), (22, 220), (20, 240), (19, 260), (17, 280), (16, 300), (15, 320), (14, 340), (13, 360), (13, 380), (12, 400), (11, 420), (11, 440), (10, 460), (10, 480), (10, 500)]
[(10000, 1), (5000, 2), (3333, 3), (2500, 4), (2000, 5), (1666, 6), (1428, 7), (1250, 8), (1111, 9), (1000, 10), (500, 20), (250, 40), (166, 60), (125, 80), (100, 100), (83, 120), (71, 140), (62, 160), (55, 180), (50, 200), (45, 220), (41, 240), (38, 260), 

In [150]:
len(params_list), len(ratings_list), len(all_params_list)

(5, 175, 175)

In [151]:
def gather_data(_N_ITEMS, _K_RESPONSES, distortion_values, exp_dir, metrics_list, _M_CATEGORIES, actual_p=False): 
    final_table = pd.DataFrame() 
    for distortion in distortion_values:
        file_path = f'{exp_dir}results_N={_N_ITEMS}_K={_K_RESPONSES}_cat_responses_simulated_distr_dist={distortion}_gen_N={_N_ITEMS}_K={_K_RESPONSES}_M={_M_CATEGORIES}_num_samples=1000.pkl.csv'
        if actual_p:
            file_path = f'{exp_dir}results_N={_N_ITEMS}_K={_K_RESPONSES}_cat_actual_responses_simulated_distr_dist={distortion}_gen_N={_N_ITEMS}_K={_K_RESPONSES}_M={_M_CATEGORIES}_num_samples=1000.pkl.csv'
        experiment_results = pd.read_csv(file_path)
        
        intermediate_table = pd.DataFrame() 
        intermediate_table['$\\Delta$'] = (experiment_results['M2 GT Alt'] - experiment_results['M1 GT Alt']).abs() 
        intermediate_table['p-value'] = experiment_results['GT_Pvalue'] 
        intermediate_table['M1 GT Alt'] = experiment_results['M1 GT Alt']
        intermediate_table['M2 GT Alt'] = experiment_results['M2 GT Alt']
        # intermediate_table['Metric'] = ['$\\Gamma_{\\rm Accuracy}$', '$\\Gamma_{\\rm F1-score}$']  
        intermediate_table['Metric'] = metrics_list 
        # intermediate_table['Metric'] = ['Accuracy'] 
        intermediate_table[f'$\\epsilon$'] = distortion 
        final_table = pd.concat([final_table, intermediate_table]) 
 
    final_table = final_table.melt(["Metric", "$\\epsilon$"]).sort_values(by=["Metric","variable"]).pivot(index = "$\\epsilon$", columns=["Metric","variable"]) 
    # final_table = final_table.reset_index(drop=True) 
    final_table = final_table.reset_index() 
    final_table.columns = pd.MultiIndex.from_tuples([(j,k) for i,j,k in final_table.columns]) 
    final_table.columns = ['\\_'.join(col) for col in final_table.columns] 
    final_table["N"] = pd.Series([_N_ITEMS]*len(distortion_values)) 
    final_table["K"] = pd.Series([_K_RESPONSES]*len(distortion_values)) 
    # final_table["NxK"] = final_table["N"]*final_table["K"] 
 
    return final_table

In [152]:
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_dices/", "dataset": "DICES", "num_categories": "3",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_d3code/", "dataset": "D3code", "num_categories": "2",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ1/", "dataset": "JobsQ1", "num_categories": "5",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_jobsQ3/", "dataset": "JobsQ3", "num_categories": "12",}
# dataset_info = {"exp_dir": "../../../../data/ptest_arr_toxicity/", "dataset": "Toxicity", "num_categories": "2",}

# dataset_info = {"exp_dir": "../ptest_arr_uniform/", "dataset": "uniform", "num_categories": "2",}
# dataset_info = {"exp_dir": "../ptest_arr_gamma/", "dataset": "gamma", "num_categories": "3",}

datasets = [{"exp_dir": "../../../../data/ptest_arr_toxicity/", "dataset": "Toxicity", "num_categories": "2",},
            {"exp_dir": "../../../../data/ptest_arr_dices/", "dataset": "DICES", "num_categories": "3",},
            {"exp_dir": "../../../../data/ptest_arr_d3code/", "dataset": "D3code", "num_categories": "2",},
            {"exp_dir": "../../../../data/ptest_arr_jobsQ1/", "dataset": "JobsQ1", "num_categories": "5",},
            {"exp_dir": "../../../../data/ptest_arr_jobsQ3/", "dataset": "JobsQ3", "num_categories": "12",},]

# datasets = [{"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=2)", "num_categories": "2",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=3)", "num_categories": "3",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=4)", "num_categories": "4",},
#             {"exp_dir": "../ptest_arr_uniform/", "dataset": "Balanced (M=5)", "num_categories": "5",},
#             {"exp_dir": "../../../../shared/rc/populaltetion/ptest_arr_uniform/", "dataset": "Balanced (M=12)", "num_categories": "12",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=2)", "num_categories": "2",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=3)", "num_categories": "3",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=4)", "num_categories": "4",},
#             {"exp_dir": "../ptest_arr_gamma/", "dataset": "Unbalanced (M=5)", "num_categories": "5",},
#             {"exp_dir": "../../../../shared/rc/population/ptest_arr_gamma/", "dataset": "Unbalanced (M=12)", "num_categories": "12",},]

In [153]:
col = "K"
distortion = 0.2
metrics_list = ['Accuracy', 'MAE', 'Wins', 'KL-Div']

confidence_level = 0.95
nk_list_idx=0

In [154]:
all_dfs = []

for dataset_info in datasets:
    _M_CATEGORIES = dataset_info['num_categories']
    exp_dir = dataset_info['exp_dir']
    dataset = dataset_info['dataset']

    errors = []
    # data_nk_list = []
    # for i,nks in enumerate(params_list):
    data_df_list = []
    for n,k in params_list[nk_list_idx][:35]:
    # for n,k in nks:
        try:
            data_df = gather_data(n, k, [distortion], exp_dir, metrics_list, _M_CATEGORIES)
            data_df_list.append(data_df)
        except FileNotFoundError:
            errors.append((n,k))
            # print(f"File not found!: {n,k}")
        except:
            errors.append((n,k))
            print(f"Some exception occured!: {n,k}")

    if data_df_list:
        df_ratings_nk = pd.concat(data_df_list)
        df_ratings_nk = df_ratings_nk.reset_index(drop=True)
        all_dfs.append(df_ratings_nk)
        # data_nk_list.append(nk_list[i])

    print(f"Distortion: {distortion}, Error list len: {len(errors)}")

    # break

print(len(all_dfs))

Distortion: 0.2, Error list len: 0
Distortion: 0.2, Error list len: 0
Distortion: 0.2, Error list len: 0
Distortion: 0.2, Error list len: 0
Distortion: 0.2, Error list len: 0
5


In [ ]:
ci_dfs = []
for dataset_info in datasets:
    exp_dir = dataset_info['exp_dir']
    
    # ci_path = f"{exp_dir}ci_2500"
    ci_path = f"{exp_dir}ci_1000"
    # ci_path = f"{exp_dir}ci"

    alt_ci_nk_dict = compress_pickle.load(f"{ci_path}/ci_level={confidence_level}_col={col}_500_dist={distortion}.pkl.lz4")
    # print(len(alt_ci_nk_dict['Accuracy']))

    ci_metric_dfs_dict = {}

    for metric in metrics_list:
        ci_rows = []
        for idx, (n_items, k_responses) in enumerate(params_list[nk_list_idx][:35]):
            # print(idx, n_items, k_responses)
            ci_lower, ci_upper = alt_ci_nk_dict[metric][nk_list_idx][idx][0]
            mean_score = alt_ci_nk_dict[metric][nk_list_idx][idx][1]
            ci_rows.append({'ci_lower':ci_lower, 'ci_upper':ci_upper, 'ci_width':ci_upper-ci_lower, 'mean_score':mean_score, 'N':n_items, 'K':k_responses})
            # break

        ci_metric_dfs_dict[metric] = pd.DataFrame(ci_rows)

    ci_dfs.append(ci_metric_dfs_dict)

print(len(ci_dfs))


5


In [125]:
# base_path = "output/tables_500"
base_path = f"output/tables_{nk_list[nk_list_idx]}"
if not os.path.exists(base_path):
    os.makedirs(base_path)

# table_name = os.path.join(base_path, "table.tex")
# df.to_latex(table_name, index=False, float_format="%.4f")

In [126]:
# for idx, dataset_info in enumerate(datasets):
#     dataset = dataset_info['dataset']
#     sub_df = all_dfs[idx][['N', 'K', 'Accuracy\\_p-value', 'Accuracy\\_$\\Delta$', 'Accuracy\\_M1 GT Alt', 'Accuracy\\_M2 GT Alt']]
#     sub_df = sub_df[sub_df['K']<=100]
#     table_name = os.path.join(base_path, f"{dataset}_accuracy_table_k_100.tex")
#     sub_df.to_latex(table_name, index=False, float_format="%.4f")

In [127]:
results_list = []
for idx, df in enumerate(all_dfs):
    results_pval = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'p-value'}
    results_k = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']}", 'Stat':'K'}
    results_delta = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']}", 'Stat':'$\\Delta$'}

    for metric in metrics_list:
        results_pval[metric] = df[f'{metric}\\_p-value'].min()
        min_p_index = df[f'{metric}\\_p-value'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results_k[metric] = int(k_value)
        results_delta[metric] = df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results_pval)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df = res_df.set_index('idx')
res_df

,Dataset,Stat,Accuracy,MAE,Wins,KL-Div
idx,,,,,,
0,Toxicity (M=12),p-value,0.247970,0.160146,0.248032,0.170061
0,Toxicity (M=12,K,1.000000,100.000000,1.000000,120.000000
0,Toxicity (M=12,$\Delta$,0.027483,0.039760,13.741742,0.019519
1,DICES (M=12),p-value,0.216788,0.081862,0.205889,0.112893
1,DICES (M=12,K,1.000000,100.000000,20.000000,240.000000
1,DICES (M=12,$\Delta$,0.034529,0.036537,5.454454,0.041947
2,D3code (M=12),p-value,0.319030,0.179608,0.320213,0.178535
2,D3code (M=12,K,1.000000,140.000000,1.000000,100.000000
2,D3code (M=12,$\Delta$,0.021359,0.040715,10.679680,0.017343


In [128]:
table_name = os.path.join(base_path, f"k_delta_low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f", multirow=True)

### K for lowest p-value

In [129]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        min_p_index = df[f'{metric}\\_p-value'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results[f'{metric}\\_K'] = int(k_value)
        results[f'{metric}\\_$\\Delta$'] = df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Accuracy\_K,Accuracy\_$\Delta$,MAE\_K,MAE\_$\Delta$,Wins\_K,Wins\_$\Delta$,KL-Div\_K,KL-Div\_$\Delta$
0,Toxicity (M=12),1,0.027483,100,0.039760,1,13.741742,120,0.019519
1,DICES (M=12),1,0.034529,100,0.036537,20,5.454454,240,0.041947
2,D3code (M=12),1,0.021359,140,0.040715,1,10.679680,100,0.017343
3,JobsQ1 (M=12),2,0.066667,80,0.032653,1,34.388388,240,0.079628
4,JobsQ3 (M=12),40,0.215132,500,0.016235,100,3.602603,500,0.095885


In [130]:
table_name = os.path.join(base_path, f"k_delta_for_low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest p-value

In [131]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        results[f'{metric}\\_p-value'] = df[f'{metric}\\_p-value'].min()
    results_list.append(results)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Accuracy\_p-value,MAE\_p-value,Wins\_p-value,KL-Div\_p-value
0,Toxicity (M=12),0.247970,0.160146,0.248032,0.170061
1,DICES (M=12),0.216788,0.081862,0.205889,0.112893
2,D3code (M=12),0.319030,0.179608,0.320213,0.178535
3,JobsQ1 (M=12),0.049684,0.018097,0.054884,0.048033
4,JobsQ3 (M=12),0.201035,0.012961,0.135127,0.054268


In [132]:
table_name = os.path.join(base_path, f"low_p_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest K for p<0.05

In [133]:
results_list = []
for idx, df in enumerate(all_dfs):
    results = {'Dataset': f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'K'}
    results_delta = {'Dataset': f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'$\\Delta$'}
    
    for metric in metrics_list:
        ls = df[f'{metric}\\_p-value']
        rows = df.iloc[ls.index[ls<0.05]]['K']
        
        if len(rows)==0:
            results[metric] = '-'
            results_delta[metric] = '-'
        else:
            results[metric] = rows.min()
            results_delta[metric] = df.iloc[rows.idxmin()][f'{metric}\\_$\\Delta$']
    results_list.append(results)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Stat,Accuracy,MAE,Wins,KL-Div
0,Toxicity (M=12),K,-,-,-,-
1,Toxicity (M=12),$\Delta$,-,-,-,-
2,DICES (M=12),K,-,-,-,-
3,DICES (M=12),$\Delta$,-,-,-,-
4,D3code (M=12),K,-,-,-,-
5,D3code (M=12),$\Delta$,-,-,-,-
6,JobsQ1 (M=12),K,2,2,-,240
7,JobsQ1 (M=12),$\Delta$,0.066667,0.024922,-,0.079628
8,JobsQ3 (M=12),K,-,80,-,-
9,JobsQ3 (M=12),$\Delta$,-,0.010635,-,-


In [134]:
table_name = os.path.join(base_path, f"low_k_for_p_lt_05_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### K for lowest ci-width

In [135]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        df = ci_dict[metric]
        min_p_index = df['ci_width'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results[f'{metric}\\_K'] = int(k_value)
        # results[f'{metric}\\_$\\Delta$'] = df.iloc[min_p_index]['mean_score']
    results_list.append(results)
    # print(results)
    # break

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Accuracy\_K,MAE\_K,Wins\_K,KL-Div\_K
0,Toxicity (M=12),1,6,280,120
1,DICES (M=12),1,8,300,240
2,D3code (M=12),1,9,360,120
3,JobsQ1 (M=12),80,7,280,480
4,JobsQ3 (M=12),1,1,320,500


In [136]:
table_name = os.path.join(base_path, f"k_for_low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

### Lowest ci-width

In [137]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results = {}
    results['Dataset'] = f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})"
    for metric in metrics_list:
        df = ci_dict[metric]
        results[f'{metric}\\_ci'] = df['ci_width'].min()
    results_list.append(results)
    # print(results)
    # break

res_df = pd.DataFrame(results_list)
res_df

,Dataset,Accuracy\_ci,MAE\_ci,Wins\_ci,KL-Div\_ci
0,Toxicity (M=12),0.11005,0.088353,2.0,0.076511
1,DICES (M=12),0.11400,0.061862,2.0,0.144936
2,D3code (M=12),0.11205,0.088889,2.0,0.061623
3,JobsQ1 (M=12),0.00000,0.036258,2.0,0.178156
4,JobsQ3 (M=12),0.08200,0.013667,0.0,0.298745


In [138]:
table_name = os.path.join(base_path, f"low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f")

In [139]:
results_list = []
for idx, ci_dict in enumerate(ci_dfs):
    results_ci = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'ci-width'}
    results_k = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'K'}
    results_delta = {'idx':idx, 'Dataset':f"{datasets[idx]['dataset']} (M={dataset_info['num_categories']})", 'Stat':'$\\Delta$'}
    p_df = all_dfs[idx]

    for metric in metrics_list:
        df = ci_dict[metric]
        results_ci[metric] = df['ci_width'].min()
        min_p_index = df['ci_width'].idxmin()
        k_value = df.iloc[min_p_index]['K']
        results_k[metric] = int(k_value)
        # print(k_value, p_df.iloc[min_p_index]['K'])

        results_delta[metric] = p_df.iloc[min_p_index][f'{metric}\\_$\\Delta$']
    results_list.append(results_ci)
    results_list.append(results_k)
    results_list.append(results_delta)
    # print(results)

res_df = pd.DataFrame(results_list)
res_df = res_df.set_index('idx')
res_df

,Dataset,Stat,Accuracy,MAE,Wins,KL-Div
idx,,,,,,
0,Toxicity (M=12),ci-width,0.110050,0.088353,2.000000,0.076511
0,Toxicity (M=12),K,1.000000,6.000000,280.000000,120.000000
0,Toxicity (M=12),$\Delta$,0.027483,0.019984,0.631632,0.019519
1,DICES (M=12),ci-width,0.114000,0.061862,2.000000,0.144936
1,DICES (M=12),K,1.000000,8.000000,300.000000,240.000000
1,DICES (M=12),$\Delta$,0.034529,0.020982,0.742743,0.041947
2,D3code (M=12),ci-width,0.112050,0.088889,2.000000,0.061623
2,D3code (M=12),K,1.000000,9.000000,360.000000,120.000000
2,D3code (M=12),$\Delta$,0.021359,0.017381,0.639640,0.016649


In [140]:
table_name = os.path.join(base_path, f"k_low_ci_nk_{nk_list[nk_list_idx]}.tex")
res_df.to_latex(table_name, index=False, float_format="%.4f", multirow=True)